In [ ]:
import os
os.makedirs("images", exist_ok=True)

# 📊 Portfolio Optimization using MPT and CVaR

## Introduction
Portfolio optimization is a financial technique used to balance risk and return. This project applies Modern Portfolio Theory (MPT) and Conditional Value at Risk (CVaR) to construct an optimal portfolio.

## Objective
- Maximize returns  
- Minimize risk  
- Compare MPT and CVaR  

In [ ]:
pip install yfinance pandas numpy matplotlib seaborn scipy


In [ ]:
import yfinance as yf
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt


In [ ]:
stocks = ["RELIANCE.NS","TCS.NS","INFY.NS","HDFCBANK.NS","ICICIBANK.NS"]

data = yf.download(stocks, start="2018-01-01", end="2025-01-01")['Close']

data.head()

In [ ]:
data

In [ ]:
data = yf.download(stocks, start="2018-01-01", end="2025-01-01")['Close'].dropna()

In [ ]:
returns = data.pct_change().dropna()
returns.head()

In [ ]:
# Remove extreme returns (optional but advanced)
returns = returns[(returns > -0.5) & (returns < 0.5)]
returns = returns.dropna()

In [ ]:
data = data.sort_index()
data = data.drop_duplicates()

In [ ]:
data = data.astype(float)

In [ ]:
normalized_data = data / data.iloc[0]

In [ ]:
normalized_data.plot(figsize=(8,5))
plt.title("Normalized Stock Prices")
plt.show()

In [ ]:
data.plot(figsize=(8,5))
plt.title("Actual Stock Prices")
plt.show()

In [ ]:
print("Any null values:", data.isnull().values.any())
print("Shape:", data.shape)
print("Date range:", data.index.min(), "to", data.index.max())

In [ ]:
data.info()

In [ ]:
mean_returns = returns.mean()
cov_matrix = returns.cov()

print(mean_returns)
print(cov_matrix)


In [ ]:
num_portfolios = 5000

results = []

for _ in range(num_portfolios):
    weights = np.random.random(len(stocks))
    weights /= np.sum(weights)
    
    portfolio_return = np.dot(weights, mean_returns)
    portfolio_risk = np.sqrt(np.dot(weights.T, np.dot(cov_matrix, weights)))
    
    sharpe = portfolio_return / portfolio_risk
    
    results.append([portfolio_return, portfolio_risk, sharpe])

In [ ]:
results = np.array(results)

In [ ]:
plt.figure(figsize=(8,5))
plt.scatter(results[:,1], results[:,0], c=results[:,2])
plt.xlabel("Risk")
plt.ylabel("Return")
plt.title("Efficient Frontier")
plt.colorbar(label="Sharpe Ratio")
plt.savefig("images/efficient_frontier.png", dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
def calculate_cvar(portfolio_returns, alpha=0.05):
    var = np.percentile(portfolio_returns, 100 * alpha)
    cvar = portfolio_returns[portfolio_returns <= var].mean()
    return var, cvar

In [ ]:
weights = np.random.random(len(stocks))
weights /= np.sum(weights)

portfolio_returns = returns.dot(weights)

var, cvar = calculate_cvar(portfolio_returns)

print("VaR:", var)
print("CVaR:", cvar)

In [ ]:
plt.figure(figsize=(8,5))
plt.hist(portfolio_returns, bins=50)

plt.axvline(var, color='red', label='VaR')
plt.axvline(cvar, color='green', label='CVaR')

plt.legend()
plt.title("CVaR Risk Visualization")
plt.savefig("images/cvar_histogram.png", dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
max_idx = np.argmax(results[:,2])

best_return = results[max_idx][0]
best_risk = results[max_idx][1]

plt.scatter(results[:,1], results[:,0], c=results[:,2])
plt.scatter(best_risk, best_return, color='red', s=100, label='Best Portfolio')

plt.xlabel("Risk")
plt.ylabel("Return")
plt.legend()
plt.savefig("images/best_portfolio.png", dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
max_idx = np.argmax(results[:,2])

# regenerate weights for best portfolio (simple approach)
best_weights = np.random.random(len(stocks))
best_weights /= np.sum(best_weights)

for stock, weight in zip(stocks, best_weights):
    print(stock, ":", round(weight, 3))

In [ ]:
print("Best Sharpe Ratio:", results[max_idx][2])

In [ ]:
import seaborn as sns

plt.figure(figsize=(6,4))
sns.heatmap(returns.corr(), annot=True, cmap="coolwarm")
plt.title("Correlation Matrix")
plt.savefig("images/correlation_heatmap.png", dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
print("MPT Risk (std dev):", best_risk)
print("CVaR Risk:", cvar)

In [ ]:
print("Expected Return:", best_return)
print("Portfolio Risk:", best_risk)

## Backtesting
Backtesting evaluates how the portfolio would perform over time using historical data.

In [ ]:
results = []
weights_record = []

for _ in range(num_portfolios):
    weights = np.random.random(len(stocks))
    weights /= np.sum(weights)
    
    weights_record.append(weights)
    
    portfolio_return = np.dot(weights, mean_returns)
    portfolio_risk = np.sqrt(np.dot(weights.T, np.dot(cov_matrix, weights)))
    
    sharpe = portfolio_return / portfolio_risk
    
    results.append([portfolio_return, portfolio_risk, sharpe])

results = np.array(results)
weights_record = np.array(weights_record)

In [ ]:
best_idx = np.argmax(results[:,2])
best_weights = weights_record[best_idx]
best_weights

In [ ]:
print("----- Optimal Portfolio Allocation -----")

for stock, weight in zip(stocks, best_weights):
    print(f"{stock}: {round(weight*100,2)}%")

## Final Observation

- MPT optimized portfolio provides better risk-adjusted returns.
- Equal-weight portfolio is simple but less efficient.
- CVaR helps identify extreme downside risks.

In [ ]:
# Equal weights portfolio
equal_weights = np.ones(len(stocks)) / len(stocks)

equal_returns = returns.dot(equal_weights)

# MPT best portfolio (Sharpe)
max_idx = np.argmax(results[:,2])
best_return = results[max_idx][0]
best_risk = results[max_idx][1]

print("Equal Portfolio Mean Return:", equal_returns.mean())
print("MPT Portfolio Return:", best_return)
print("MPT Portfolio Risk:", best_risk)

In [ ]:
portfolio_returns = returns.dot(equal_weights)

var, cvar = calculate_cvar(portfolio_returns)

print("Mean Return:", portfolio_returns.mean())
print("Standard Risk:", portfolio_returns.std())
print("VaR:", var)
print("CVaR:", cvar)

In [ ]:
rolling_cvar = portfolio_returns.rolling(window=100).apply(
    lambda x: calculate_cvar(x)[1]
)

plt.figure(figsize=(8,5))
plt.plot(rolling_cvar)
plt.title("Rolling CVaR (Risk Over Time)")
plt.savefig("images/rolling_cvar.png", dpi=150, bbox_inches='tight')
plt.show()

## Comparison of Strategies
We compare equal-weight portfolio, MPT optimized portfolio, and CVaR risk to understand performance differences.

In [ ]:
# Equal weights portfolio
equal_weights = np.ones(len(stocks)) / len(stocks)

equal_returns = returns.dot(equal_weights)

# MPT best portfolio
max_idx = np.argmax(results[:,2])
best_return = results[max_idx][0]
best_risk = results[max_idx][1]

print("------ COMPARISON ------")
print("Equal Return:", equal_returns.mean())
print("Equal Risk:", equal_returns.std())

print("MPT Return:", best_return)
print("MPT Risk:", best_risk)

print("CVaR:", cvar)

## Key Insights

- Diversification reduces portfolio risk.
- MPT provides optimal risk-return balance.
- CVaR captures extreme losses better than variance.
- MPT portfolios generally perform better than equal-weight portfolios.
- CVaR is useful for downside risk protection.

## Conclusion

This project demonstrates that portfolio optimization using MPT improves return efficiency, while CVaR enhances risk management by focusing on extreme losses. Combining both provides a more robust investment strategy.